# 04, the random split

A random split is the first evaluation many projects reach for: shuffle every labelled CVE, hold a share out, score the model on it. The plan forbids it (config/splits.yaml splits by publication date, and the floor is chosen on 2024 and judged on 2025 and later), because a random holdout puts CVEs from the same months, the same filers and the same bursts of near identical advisories on both sides. This notebook fits the same band floor both ways, on the same number of rows, compares the two scores, and then scores both models on the same 2024 CVEs, to separate what the split changes from what the population changes.

In [1]:
import json

import numpy as np
import pandas as pd
import polars as pl

pl.Config.set_tbl_rows(12)
pl.Config.set_tbl_width_chars(110)
SEED = 20261114  # evaluation/phase2_plan.yaml, defaults.seed
print("numpy", np.__version__, "pandas", pd.__version__, "polars", pl.__version__)

numpy 2.5.4 pandas 2.3.3 polars 2.1.0


In [2]:
from unscheduled_models import floors
from unscheduled_models.features import Features

record = json.load(open("results/models/floors.json"))
chosen = record["chosen"]["band"]
variant, c = record["representation"]["chosen"], chosen["C"]
print(f"the band floor as chosen: {variant} grams, C {c}, development kappa {chosen['dev']['qwk']:.3f}")

def labelled(split):
    inputs = pl.read_parquet(f"data/sets/{split}_inputs.parquet")
    labels = pl.read_parquet(f"data/sets/{split}_labels.parquet").select("cve_id", "band")
    return inputs.join(labels, on="cve_id").filter(pl.col("band").is_not_null())

train, dev = labelled("train"), labelled("development")
print(f"{train.height:,} train and {dev.height:,} development CVEs with an NVD band")

the band floor as chosen: word_char grams, C 0.3, development kappa 0.669
101,427 train and 23,272 development CVEs with an NVD band


Both arms fit on 50,000 CVEs and score on as many CVEs as development holds. The time arm draws its 50,000 from train (published before 2024) and scores on development (2024). The random arm pools train and development, holds out a random set the size of development, and draws its 50,000 from the rest.

In [3]:
N = 50_000

def fit(rows):
    features = Features(variant)
    model, _ = floors.fit(features.fit_transform(rows), rows["band"].to_list(), c, SEED)
    return features, model

def kappa(fitted, rows):
    features, model = fitted
    guess = model.classes_[np.argmax(model.predict_proba(features.transform(rows)), axis=1)]
    return floors.score("qwk", rows["band"].to_list(), [str(g) for g in guess])

time_model = fit(train.sample(n=N, seed=SEED))
time_kappa = kappa(time_model, dev)

pool = pl.concat([train.select(dev.columns), dev]).sample(fraction=1.0, shuffle=True, seed=SEED)
held, rest = pool.head(dev.height), pool.tail(pool.height - dev.height)
random_model = fit(rest.sample(n=N, seed=SEED))
random_kappa = kappa(random_model, held)
print(f"split by time, scored on development: kappa {time_kappa:.3f}")
print(f"random split, scored on its holdout:  kappa {random_kappa:.3f}")
print(f"difference, random less time:         {random_kappa - time_kappa:+.3f}")

split by time, scored on development: kappa 0.666
random split, scored on its holdout:  kappa 0.657
difference, random less time:         -0.009


The two holdouts are different CVEs. Where the random holdout's CVEs come from, by publication year, against development, which is all 2024, and how often each year's CVEs carried a filer band on the day they were published. pandas does the counting.

In [4]:
held_pd = held.select("published_year", "filer_band").to_pandas()
years = pd.DataFrame(
    {
        "random holdout": held_pd["published_year"].value_counts(normalize=True).sort_index() * 100,
        "development": dev["published_year"].to_pandas().value_counts(normalize=True).sort_index() * 100,
        "filer band at day 0": held_pd.groupby("published_year")["filer_band"].apply(lambda s: s.notna().mean() * 100),
    }
).fillna(0.0)
years.tail(8).round(1)

                random holdout  development  filer band at day 0
published_year                                                  
2017                       1.4          0.0                  0.0
2018                       1.4          0.0                  0.0
2019                       7.2          0.0                  0.0
2020                      14.4          0.0                  0.0
2021                      14.9          0.0                  0.0
2022                      19.0          0.0                  0.0
2023                      21.5          0.0                 10.2
2024                      18.9        100.0                 59.9

The same rows for both: the 2024 CVEs inside the random holdout. The time arm's model saw no 2024 CVE. The random arm's model saw the other 2024 CVEs of the pool, from the same months, the same filers and the same bursts.

In [5]:
same = held.filter(pl.col("published_year") == 2024)
on_same = {"time split": kappa(time_model, same), "random split": kappa(random_model, same)}
print(f"{same.height:,} CVEs from 2024 in the random holdout")
for name, value in on_same.items():
    print(f"{name:>12}: kappa {value:.3f}")
print(f"  difference: {on_same['random split'] - on_same['time split']:+.3f}")

4,406 CVEs from 2024 in the random holdout
  time split: kappa 0.664
random split: kappa 0.722
  difference: +0.057


Read together, the three results show what the random split hides. Scored on its own holdout it looks no better than the split by time, even a little worse, because that holdout is a different population: mostly CVEs from years when a filer seldom gave a band on the day of publication, which leaves the floor the text alone. Scored on the same 2024 CVEs, the random split's model is ahead, because it was fit on other CVEs from 2024, from the same months and filers, and from the first year in which most CVEs carried a filer band at publication. That lead is credit a random split hands the model and no deployment pays out: on the day a new CVE is scored, the CVEs published around it are not labelled yet. The build chooses every model on development and judges it once on test, both later than everything it was fit on (evaluation/phase2_plan.yaml, windows).